<a href="https://colab.research.google.com/github/tadeugomes/2_encontro_enap/blob/main/Aula5_Exercicio5_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercício 5.1 - Solução

In [1]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

Authenticated


In [4]:
import pandas as pd
## Defina o id do seu projeto no bigquery!!!!!
project_id = 'enap-mba-470912' # Defina o id do seu projeto no bigquery!!!!!
## Defina o id do seu projeto no bigquery!!!!!

df = pd.io.gbq.read_gbq('''
SELECT
  pib.ano AS ano,
  uf.sigla AS sigla_uf,
  mun.id_municipio,
  pop.populacao,
  mun.nome AS nome_municipio,
  pib.pib,
  ROUND(CAST(pib.pib AS FLOAT64)/NULLIF(CAST(pop.populacao AS FLOAT64),0), 6) AS pibpercapita
FROM
  `basedosdados.br_ibge_pib.municipio` AS pib
JOIN
  `basedosdados.br_ibge_populacao.municipio` AS pop
    ON pib.id_municipio = pop.id_municipio AND pib.ano = pop.ano
JOIN
  `basedosdados.br_bd_diretorios_brasil.municipio` AS mun
    ON pib.id_municipio = mun.id_municipio
JOIN
  `basedosdados.br_bd_diretorios_brasil.uf` AS uf
    ON mun.id_uf = uf.id_uf
WHERE
  pib.ano BETWEEN 2002 AND 2018
ORDER BY
  ano, sigla_uf, nome_municipio
''', project_id=project_id)

df.head()

/tmp/ipython-input-679519711.py:6: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  df = pd.io.gbq.read_gbq('''


,ano,sigla_uf,id_municipio,populacao,nome_municipio,pib,pibpercapita
0,2002,AC,1200013,8454,Acrelândia,39622000,4686.775491
1,2002,AC,1200054,3611,Assis Brasil,12789000,3541.678205
2,2002,AC,1200104,17649,Brasiléia,72997000,4136.041702
3,2002,AC,1200138,6382,Bujari,26743000,4190.379191
4,2002,AC,1200179,5814,Capixaba,32492000,5588.579291


In [1]:
import pandas as pd

# Defina o id do seu projeto no BigQuery
project_id = "enap-mba-470912"

query = """
WITH pib_pop AS (
  SELECT
    pib.ano,
    uf.sigla AS sigla_uf,
    mun.id_municipio,
    pop.populacao,
    mun.nome AS nome_municipio,
    pib.pib,
    ROUND(CAST(pib.pib AS FLOAT64)/NULLIF(CAST(pop.populacao AS FLOAT64),0), 6) AS pibpercapita
  FROM `basedosdados.br_ibge_pib.municipio` AS pib
  JOIN `basedosdados.br_ibge_populacao.municipio` AS pop
    ON pib.id_municipio = pop.id_municipio AND pib.ano = pop.ano
  JOIN `basedosdados.br_bd_diretorios_brasil.municipio` AS mun
    ON pib.id_municipio = mun.id_municipio
  JOIN `basedosdados.br_bd_diretorios_brasil.uf` AS uf
    ON mun.id_uf = uf.id_uf
  WHERE pib.ano BETWEEN 2015 AND 2018
),
ideb_agg AS (
  SELECT
    ano,
    id_municipio,
    AVG(CASE WHEN LOWER(rede) = 'estadual'  THEN ideb END) AS ideb_estadual,
    AVG(CASE WHEN LOWER(rede) = 'federal'   THEN ideb END) AS ideb_federal,
    AVG(CASE WHEN LOWER(rede) = 'municipal' THEN ideb END) AS ideb_municipal,
    AVG(CASE WHEN LOWER(rede) = 'publica'   THEN ideb END) AS ideb_publica
  FROM `basedosdados.br_inep_ideb.municipio`
  WHERE ano BETWEEN 2015 AND 2018
  GROUP BY ano, id_municipio
)
SELECT
  p.ano, p.sigla_uf, p.id_municipio, p.nome_municipio,
  p.populacao, p.pib, p.pibpercapita,
  COALESCE(i.ideb_municipal,0) AS ideb_municipal,
  COALESCE(i.ideb_estadual,0)  AS ideb_estadual,
  COALESCE(i.ideb_federal,0)   AS ideb_federal,
  COALESCE(i.ideb_publica,0)   AS ideb_publica
FROM pib_pop p
LEFT JOIN ideb_agg i
  ON p.ano = i.ano AND p.id_municipio = i.id_municipio
ORDER BY p.ano, p.sigla_uf, p.nome_municipio
"""

df_saeb = pd.io.gbq.read_gbq(query, project_id=project_id)
df_saeb.head()

/tmp/ipython-input-3976890042.py:50: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  df_saeb = pd.io.gbq.read_gbq(query, project_id=project_id)


,ano,sigla_uf,id_municipio,nome_municipio,populacao,pib,pibpercapita,ideb_municipal,ideb_estadual,ideb_federal,ideb_publica
0,2015,AC,1200013,Acrelândia,13869,212961000,15355.180619,5.0,4.3,0.0,0.0
1,2015,AC,1200054,Assis Brasil,6738,77234000,11462.451766,4.4,4.0,0.0,0.0
2,2015,AC,1200104,Brasiléia,23849,388114000,16273.806030,5.2,6.5,0.0,0.0
3,2015,AC,1200138,Bujari,9339,147868000,15833.386872,4.3,3.4,0.0,0.0
4,2015,AC,1200179,Capixaba,10498,160134000,15253.762621,4.5,4.2,0.0,0.0


## Codigo comentado

O WITH ... SELECT que você executou faz, em três passos, a junção entre indicadores socioeconômicos (PIB e população) e indicadores educacionais (IDEB) no nível de município e ano:

CTE pib_pop – Essa subconsulta pega o Produto Interno Bruto (PIB) municipal e a população correspondentes, juntando as tabelas públicas da Base dos Dados. Ela seleciona o ano (pib.ano), a sigla da unidade federativa (uf.sigla), o código do município (id_municipio), o nome do município, a população e o PIB. Em seguida calcula o PIB per capita dividindo o PIB pela população (usando NULLIF para evitar divisão por zero) e restringe o período de 2015 a 2018.

CTE ideb_agg – Esta subconsulta lê a tabela do Ideb municipal e agrega as notas por rede de ensino. Para cada município e ano são calculadas quatro médias com AVG(...) e CASE WHEN LOWER(rede) = '...': – uma para a rede estadual, outra para a federal, outra para a municipal e outra para a rede pública. O uso de LOWER(rede) garante que a comparação é insensível a maiúsculas/minúsculas, de acordo com os valores retornados pela consulta SELECT DISTINCT rede. O GROUP BY ano, id_municipio assegura que cada linha agregada corresponda a um par ano–município.

Consulta final – Faz um LEFT JOIN entre pib_pop e ideb_agg usando ano e id_municipio. Isso mantém todos os municípios da base de PIB mesmo quando não há Ideb disponível para alguma rede (caso em que COALESCE substitui valores NULL por 0). O resultado inclui, para cada município e ano, o PIB, a população, o PIB per capita e as médias do Ideb por rede, ordenado por ano, estado e nome do município.

Esse tipo de junção permite comparar desempenho econômico e educacional dos municípios: o Saeb (avaliações de larga escala que alimentam o Ideb) é aplicado periodicamente
basedosdados.org
 e suas médias estruturam o Índice de Desenvolvimento da Educação Básica
basedosdados.org
. Assim, o dataset final combina os indicadores educacionais provenientes do Saeb/Ideb com os dados de PIB per capita para cada município no período 2015‑2018.

## Solução do Exercício 5.2


In [4]:
import pandas as pd

# Defina seu project_id (o mesmo usado no SQL)
project_id = "enap-mba-470912"
location = "us"  # região do dataset na Base dos Dados

# Consulta SQL para PIB, população e PIB per capita
query_pib = """
SELECT
  pib.ano,
  uf.sigla AS sigla_uf,
  mun.id_municipio,
  pop.populacao,
  mun.nome AS nome_municipio,
  pib.pib,
  ROUND(CAST(pib.pib AS FLOAT64)/NULLIF(CAST(pop.populacao AS FLOAT64), 0), 6) AS pibpercapita
FROM `basedosdados.br_ibge_pib.municipio` AS pib
JOIN `basedosdados.br_ibge_populacao.municipio` AS pop
  ON pib.id_municipio = pop.id_municipio AND pib.ano = pop.ano
JOIN `basedosdados.br_bd_diretorios_brasil.municipio` AS mun
  ON pib.id_municipio = mun.id_municipio
JOIN `basedosdados.br_bd_diretorios_brasil.uf` AS uf
  ON mun.id_uf = uf.id_uf
WHERE pib.ano BETWEEN 2015 AND 2018
"""

# Consulta SQL para IDEB
query_ideb = """
SELECT
  ano,
  id_municipio,
  LOWER(rede) AS rede,
  ideb
FROM `basedosdados.br_inep_ideb.municipio`
WHERE ano BETWEEN 2015 AND 2018
"""

# Ler PIB/população no BigQuery (necessário instalar pandas-gbq)
df_pib = pd.read_gbq(query_pib, project_id=project_id, location=location)

# Ler Ideb por município no BigQuery
df_ideb_raw = pd.read_gbq(query_ideb, project_id=project_id, location=location)

# Pivotar por rede de ensino
df_ideb_pivot = (
    df_ideb_raw
    .pivot_table(
        index=['ano', 'id_municipio'],
        columns='rede',
        values='ideb',
        aggfunc='mean',
        dropna=False
    )
    .reset_index()
    .fillna(0)  # substituir ausências por zero
)

# Renomear as colunas pivotadas com prefixo
df_ideb_pivot.columns.name = None
rename_map = {col: f"ideb_{col}" for col in df_ideb_pivot.columns if col not in ['ano', 'id_municipio']}
df_ideb_pivot = df_ideb_pivot.rename(columns=rename_map)

# Fazer o merge com PIB/população
ideb_cols = [col for col in df_ideb_pivot.columns if col.startswith('ideb_')]
df_final = (
    df_pib.merge(df_ideb_pivot, on=['ano', 'id_municipio'], how='left')
    .fillna({col: 0 for col in ideb_cols})
)

# df_final contém: ano, sigla_uf, id_municipio, nome_municipio,
# populacao, pib, pibpercapita, ideb_estadual, ideb_federal, ideb_municipal, ideb_publica
print(df_final.head())


/tmp/ipython-input-1474020619.py:39: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  df_pib = pd.read_gbq(query_pib, project_id=project_id, location=location)
/tmp/ipython-input-1474020619.py:42: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  df_ideb_raw = pd.read_gbq(query_ideb, project_id=project_id, location=location)


    ano sigla_uf id_municipio  populacao         nome_municipio         pib  \
0  2015       RO      1100015      25578  Alta Floresta D'Oeste   421300000   
1  2016       RO      1100015      25506  Alta Floresta D'Oeste   478217000   
2  2017       RO      1100015      25437  Alta Floresta D'Oeste   485374000   
3  2018       RO      1100015      23167  Alta Floresta D'Oeste   498980000   
4  2015       RO      1100023     104401              Ariquemes  2037799000   

   pibpercapita  ideb_estadual  ideb_federal  ideb_municipal  ideb_publica  
0  16471.186176       5.000000           0.0             5.0      0.000000  
1  18749.196268       0.000000           0.0             0.0      0.000000  
2  19081.416834       5.066667           0.0             4.7      4.966667  
3  21538.395131       0.000000           0.0             0.0      0.000000  
4  19518.960546       4.950000           0.0             4.8      0.000000  


## [Demonstração](https://drive.google.com/file/d/1PYtx7LSVGYzv3pcjPY6VB8uO7P1Cl0cF/view?usp=sharing) de como criar um dataset

## Submeta o seu dataframe criando uma tabela no BigQuery

In [6]:
# Enviar o DataFrame para o BigQuery
try:
    df_final.to_gbq(
        "enapcd2021.pibpercapita",
        project_id=project_id,
        chunksize=40000,
        if_exists='replace'
    )
    print("Dados enviados com sucesso para a tabela 'enapcd2021.pibpercapita'")
except Exception as e:
    print(f"Erro ao enviar dados para o BigQuery: {e}")
    print("\nVerifique:")
    print("1. Se o dataset 'enapcd2021' existe no seu projeto")
    print("2. Se você tem permissão para criar tabelas")
    print("3. Se o nome da tabela está correto")
    print("4. Sua autenticação está funcionando corretamente")

/tmp/ipython-input-1657243520.py:3: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df_final.to_gbq(
100%|██████████| 1/1 [00:00<00:00, 9754.20it/s]

Dados enviados com sucesso para a tabela 'enapcd2021.pibpercapita'
